# 01 - Benchmark- und Ground-Truth-Erstellung

Dieses Notebook erzeugt die kontrollierten Benchmark-Datensätze und die zugehörigen
Ground-Truth-Daten für alle vier Teilforschungsfragen (vgl. Abschnitt 3.2.3 und 3.2.4
der Arbeit). Es wird **vor** allen Experiment-Notebooks ausgeführt und ist die einzige
Stelle, an der Zufallsprozesse (Seed) für die Benchmark-Erstellung verwendet werden,
damit alle nachfolgenden Experimente auf exakt denselben Daten arbeiten.

**Strategie je Fehlerart (Begründung siehe Abschnitt 3.2.3):**

| Fehlerart | Ground-Truth-Quelle |
|---|---|
| TF1 Duplikate | Reale URL-Duplikate + kontrolliert generierte synthetische Duplikat-Paare |
| TF2 Fehlende Werte | Kontrolliertes Maskieren real bekannter Werte (Missing-Completely-at-Random) |
| TF3 Formatierungsfehler | Real: Rohwert (`rfd_main.csv`) vs. Referenzwert (`rfd_main_cleaned.csv`) |
| TF4 Semantische Heterogenität | Mehrheitsvotum-Mapping `thread_category` -> `parent_category`, manuell geprüft |

**Ausgabeordner:** `benchmark/` (Input-Dateien für die Methoden) und je Ground-Truth
eine separate `_groundtruth.csv`-Datei, damit die wahren Werte den Methoden (klassisch
und LLM) nicht versehentlich zugänglich sind.


## 0. Setup

**Wichtiger Hinweis zur Datenaufbereitung:** Die Spalte `Unnamed: 0` in beiden
CSV-Dateien ist **kein** eindeutiger Zeilenindex, sondern ein Artefakt der
Kaggle-Aufbereitung (sie zählt nur 0-29 im Kreis, vermutlich ein rekonstruierter
Tagesindex). Sie wird verworfen. Stattdessen wird jeder Datei ein eigener, echter
Positions-Index (`row_id`, 0..n-1) zugewiesen. Da der Referenzdatensatz eine Zeile
weniger enthält als der Rohdatensatz (1325 statt 1326), ist der Positions-Index
**zwischen** den beiden Dateien nicht direkt vergleichbar - dafür wird in Abschnitt 3
ein inhaltlicher Schlüssel (Titel + Autor + Zeitstempel) verwendet, um Zeilen aus
Roh- und Referenzdatensatz einander eindeutig zuzuordnen (vgl. Abschnitt 3.2.4,
Validierung der Ground-Truth-Daten).


In [1]:
import pandas as pd
import numpy as np
import re
import os

SEED = 42
rng = np.random.default_rng(SEED)

os.makedirs("benchmark", exist_ok=True)
os.makedirs("results", exist_ok=True)

df_raw = pd.read_csv("data/rfd_main.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)
df_raw["row_id"] = df_raw.index

df_clean = pd.read_csv("data/rfd_main_cleaned.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)
df_clean["row_id"] = df_clean.index

print(f"Rohdatensatz:        {df_raw.shape}")
print(f"Referenzdatensatz:   {df_clean.shape}")


Rohdatensatz:        (1326, 15)
Referenzdatensatz:   (1325, 15)


## 0.1 Zeilen-Zuordnung zwischen Roh- und Referenzdatensatz

Für TF3 (Formatierungsfehler) wird für dieselbe reale Zeile sowohl der Rohwert als
auch der Referenzwert benötigt. Da kein gemeinsamer eindeutiger Schlüssel existiert,
wird ein inhaltlicher Verbund über `title` + `author` + auf die Minute gerundeten
`creation_date`-Zeitstempel gebildet. Nur Schlüssel, die auf **beiden** Seiten genau
einmal vorkommen (eindeutige 1:1-Zuordnung), werden übernommen - alle anderen
(z.B. durch reale Duplikate) werden verworfen, um Verfälschungen der Ground Truth
auszuschließen.


In [2]:
def format_like_raw(dt):
    """Formatiert ein datetime-Objekt wie die Rohdaten-Zeitstempel, z.B. 'Jul 12th, 2020 8:09 pm'."""
    day = dt.day
    suf = "th" if 11 <= day % 100 <= 13 else {1: "st", 2: "nd", 3: "rd"}.get(day % 10, "th")
    hour12 = dt.hour % 12
    hour12 = 12 if hour12 == 0 else hour12
    ampm = "am" if dt.hour < 12 else "pm"
    return f"{dt.strftime('%b')} {day}{suf}, {dt.year} {hour12}:{dt.strftime('%M')} {ampm}"

def parse_raw_date(s):
    """Parst das Rohformat 'Mon Dth, YYYY H:MM am/pm' bzw. 'Mon D, YYYY' zu datetime
    (Ordinalsuffix wird vorher entfernt)."""
    if pd.isna(s):
        return pd.NaT
    s2 = re.sub(r"(\d+)(st|nd|rd|th)", r"\1", str(s))
    for fmt in ("%b %d, %Y %I:%M %p", "%b %d, %Y"):
        try:
            return pd.to_datetime(s2, format=fmt)
        except ValueError:
            continue
    return pd.NaT

def join_key(df, dt_series):
    return (df["title"].astype(str).str.strip().str.lower() + "||" +
            df["author"].astype(str).str.strip().str.lower() + "||" +
            dt_series.dt.floor("min").astype(str))

raw_dt = df_raw["creation_date"].apply(parse_raw_date)
clean_dt = pd.to_datetime(df_clean["creation_date"], errors="coerce")

raw_key = join_key(df_raw, raw_dt)
clean_key = join_key(df_clean, clean_dt)

raw_key_counts = raw_key.value_counts()
clean_key_counts = clean_key.value_counts()
unique_raw_keys = set(raw_key_counts[raw_key_counts == 1].index)
unique_clean_keys = set(clean_key_counts[clean_key_counts == 1].index)
common_keys = unique_raw_keys & unique_clean_keys

raw_lookup = pd.DataFrame({"row_id": df_raw["row_id"], "_key": raw_key})
clean_lookup = pd.DataFrame({"row_id": df_clean["row_id"], "_key": clean_key})

row_mapping = raw_lookup[raw_lookup["_key"].isin(common_keys)].merge(
    clean_lookup[clean_lookup["_key"].isin(common_keys)], on="_key",
    suffixes=("_raw", "_clean"))[["row_id_raw", "row_id_clean"]]

row_mapping.to_csv("benchmark/raw_clean_row_mapping.csv", index=False)
print(f"Eindeutig zugeordnete Zeilen (Roh <-> Referenz): {len(row_mapping)} von {len(df_raw)} "
      f"({len(row_mapping)/len(df_raw)*100:.1f}%)")
print("Gespeichert: benchmark/raw_clean_row_mapping.csv")


Eindeutig zugeordnete Zeilen (Roh <-> Referenz): 1293 von 1326 (97.5%)
Gespeichert: benchmark/raw_clean_row_mapping.csv


## 1. TF1 - Duplikate: Benchmark mit bekannten Duplikat-/Nicht-Duplikat-Paaren

Im Rohdatensatz gibt es zu wenige natürliche, sicher gelabelte Duplikate, um einen
Klassifikator (XGBoost) überhaupt trainieren zu können. Daher wird ein kontrollierter
Benchmark aus zwei Quellen gebaut:

1. **Reale Duplikate:** Zeilen mit identischer, nicht-leerer `url` (sicherstes reales Signal).
2. **Synthetische Duplikate:** Zufällig gezogene Zeilen werden als "erneut gepostetes"
   Duplikat leicht verändert (Tippfehler im Titel, verschobenes Datum, ggf. anderer Autor,
   ggf. andere Preis-Schreibweise) - simuliert einen realistischen Re-Post im Forum.

Alle Zeilen (real + synthetisch) werden in einen gemeinsamen Pool geschrieben. Aus den
Duplikat-Gruppen werden dann Paare erzeugt: Paare aus derselben Gruppe = Label 1,
zufällige Paare aus verschiedenen Gruppen = Label 0.


In [3]:
def light_typo(text):
    """Fügt einer zufälligen Wortstelle im Titel eine kleine Tippfehler-Störung hinzu."""
    words = str(text).split(" ")
    if len(words) > 2:
        i = rng.integers(0, len(words))
        w = words[i]
        if len(w) > 3:
            j = rng.integers(1, len(w) - 1)
            w = w[:j] + w[j + 1:]
        words[i] = w
    return " ".join(words)


In [4]:
# --- 1.1 Pool aufbauen: reale Zeilen + synthetische Duplikat-Zwillinge ---
pool_rows = []
group_id_counter = 0

for _, row in df_raw.iterrows():
    r = row.to_dict()
    r["group_id"] = group_id_counter
    r["is_synthetic"] = False
    pool_rows.append(r)
    group_id_counter += 1

pool = pd.DataFrame(pool_rows)
# row_id ist bereits eindeutig (0..1325); group_id vorerst = row_id (jede Zeile eigene Gruppe)
pool["group_id"] = pool["row_id"]

# --- 1.2 Reale Duplikate: identische, nicht-leere URL -> gemeinsame Gruppe ---
url_groups = df_raw[df_raw["url"].notna()].groupby("url")["row_id"].apply(list)
url_groups = url_groups[url_groups.apply(len) > 1]
for ids in url_groups:
    canonical = min(ids)
    pool.loc[pool["row_id"].isin(ids), "group_id"] = canonical
print(f"Reale URL-Duplikat-Gruppen: {len(url_groups)} (betrifft {sum(len(x) for x in url_groups)} Zeilen)")

# --- 1.3 Synthetische Duplikate erzeugen ---
n_synth = 150
sample_ids = rng.choice(df_raw["row_id"].values, size=n_synth, replace=False)
next_row_id = pool["row_id"].max() + 1
synth_rows = []

for rid in sample_ids:
    orig = df_raw[df_raw["row_id"] == rid].iloc[0].to_dict()
    twin = dict(orig)
    twin["row_id"] = next_row_id
    twin["group_id"] = pool.loc[pool["row_id"] == rid, "group_id"].values[0]
    twin["is_synthetic"] = True
    twin["title"] = light_typo(orig["title"])

    parsed = parse_raw_date(orig["creation_date"])
    if pd.notna(parsed):
        shift = pd.Timedelta(hours=int(rng.integers(1, 72)))
        twin["creation_date"] = format_like_raw(parsed + shift)

    if rng.random() < 0.5:
        twin["author"] = rng.choice(df_raw["author"].dropna().unique())

    if rng.random() < 0.3 and pd.notna(orig["price"]):
        p = str(orig["price"])
        twin["price"] = p if p.startswith("$") else f"${p}"

    if rng.random() < 0.4 and pd.notna(orig.get("url")):
        twin["url"] = str(orig["url"]) + f"?ref=repost{next_row_id}"

    synth_rows.append(twin)
    next_row_id += 1

pool = pd.concat([pool, pd.DataFrame(synth_rows)], ignore_index=True)
print(f"Synthetische Duplikat-Zwillinge erzeugt: {len(synth_rows)}")
print(f"Gesamter Pool: {pool.shape[0]} Zeilen, {pool['group_id'].nunique()} Gruppen")


Reale URL-Duplikat-Gruppen: 25 (betrifft 50 Zeilen)


Synthetische Duplikat-Zwillinge erzeugt: 150
Gesamter Pool: 1476 Zeilen, 1301 Gruppen


In [5]:
# --- 1.4 Paare erzeugen: positive (gleiche Gruppe) und negative (verschiedene Gruppen) ---
pos_pairs = []
for gid, members in pool.groupby("group_id")["row_id"].apply(list).items():
    if len(members) > 1:
        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                pos_pairs.append((members[i], members[j], 1))

n_neg = min(len(pos_pairs) * 3, 3000)
group_of = pool.set_index("row_id")["group_id"].to_dict()
all_ids = pool["row_id"].values
neg_pairs = set()
attempts = 0
while len(neg_pairs) < n_neg and attempts < n_neg * 20:
    a, b = rng.choice(all_ids, size=2, replace=False)
    attempts += 1
    if group_of[a] != group_of[b]:
        pair = (min(a, b), max(a, b))
        neg_pairs.add(pair)

pairs_df = pd.DataFrame(pos_pairs, columns=["row_id_a", "row_id_b", "label"])
neg_df = pd.DataFrame([(a, b, 0) for a, b in neg_pairs], columns=["row_id_a", "row_id_b", "label"])
pairs_df = pd.concat([pairs_df, neg_df], ignore_index=True).drop_duplicates(subset=["row_id_a", "row_id_b"])

# --- 1.5 Train/Val/Test-Split (stratifiziert nach Label, fester Seed) ---
from sklearn.model_selection import train_test_split
train_val, test = train_test_split(pairs_df, test_size=0.2, stratify=pairs_df["label"], random_state=SEED)
train, val = train_test_split(train_val, test_size=0.25, stratify=train_val["label"], random_state=SEED)
pairs_df["split"] = "train"
pairs_df.loc[val.index, "split"] = "val"
pairs_df.loc[test.index, "split"] = "test"

print(pairs_df["label"].value_counts())
print(pairs_df["split"].value_counts())

pool.to_csv("benchmark/tf1_duplicate_pool.csv", index=False)
pairs_df.to_csv("benchmark/tf1_duplicate_pairs.csv", index=False)
print("Gespeichert: benchmark/tf1_duplicate_pool.csv, benchmark/tf1_duplicate_pairs.csv")


label
0    552
1    184
Name: count, dtype: int64
split
train    441
test     148
val      147
Name: count, dtype: int64
Gespeichert: benchmark/tf1_duplicate_pool.csv, benchmark/tf1_duplicate_pairs.csv


## 2. TF2 - Fehlende Werte: kontrolliertes Maskieren mit bekannter Wahrheit

Im Referenzdatensatz sind fehlende Werte laut EDA **nicht** behoben worden - es gibt
also keine reale Möglichkeit, die "wahren" fehlenden Werte des Rohdatensatzes zu
kennen. Stattdessen wird ein klassischer Missing-Data-Benchmark aufgebaut: Von den
Zeilen mit **bekanntem** Zielwert wird ein Teil absichtlich maskiert (Missing
Completely at Random), der wahre Wert wird separat gespeichert (Ground Truth), und
nur die maskierte Version wird den Methoden übergeben.

Zwei Zielattribute (numerisch und kategorial, gemäß Abschnitt 3.6.1):
- **numerisch:** `price` (aus dem bereits formatbereinigten Referenzdatensatz, damit
  Formatierungsfehler und fehlende Werte sauber getrennt bleiben, vgl. Abschnitt 3.2.3)
- **kategorial:** `parent_category` (aus dem Rohdatensatz, dort unverändert gegenüber der Referenz)


In [6]:
def build_missing_benchmark(base_df, target_col, context_cols, out_prefix,
                             train_frac=0.70, val_frac=0.15, seed=SEED):
    known = base_df[base_df[target_col].notna()].copy()
    known_ids = known["row_id"].values
    rng_local = np.random.default_rng(seed)
    shuffled = rng_local.permutation(known_ids)

    n = len(shuffled)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_ids = set(shuffled[:n_train])
    val_ids = set(shuffled[n_train:n_train + n_val])
    test_ids = set(shuffled[n_train + n_val:])

    out = base_df[["row_id", target_col] + context_cols].copy()
    out = out[out["row_id"].isin(known_ids)]

    def split_of(rid):
        if rid in train_ids: return "train"
        if rid in val_ids: return "val"
        return "test"
    out["split"] = out["row_id"].apply(split_of)

    groundtruth = out[out["split"].isin(["val", "test"])][["row_id", target_col]].rename(
        columns={target_col: f"{target_col}_true"})

    out_input = out.copy()
    out_input.loc[out_input["split"].isin(["val", "test"]), target_col] = np.nan
    out_input = out_input.rename(columns={target_col: f"{target_col}_input"})

    out_input.to_csv(f"benchmark/{out_prefix}.csv", index=False)
    groundtruth.to_csv(f"benchmark/{out_prefix}_groundtruth.csv", index=False)
    print(f"{out_prefix}: {len(out_input)} Zeilen total, "
          f"{len(groundtruth)} maskiert (val+test), Split: "
          f"{out_input['split'].value_counts().to_dict()}")
    return out_input, groundtruth

context_numeric = ["views", "votes", "replies", "parent_category", "thread_category", "source"]
_ = build_missing_benchmark(df_clean, "price", context_numeric, "tf2_missing_price")

context_categorical = ["views", "votes", "replies", "price", "saving", "thread_category", "source", "title"]
_ = build_missing_benchmark(df_raw, "parent_category", context_categorical, "tf2_missing_parent_category")


tf2_missing_price: 1084 Zeilen total, 326 maskiert (val+test), Split: {'train': 758, 'test': 164, 'val': 162}
tf2_missing_parent_category: 825 Zeilen total, 248 maskiert (val+test), Split: {'train': 577, 'test': 125, 'val': 123}


## 3. TF3 - Formatierungsfehler: reale Roh-/Referenz-Paare als Ground Truth

Hier ist **kein** synthetischer Fehler nötig: Der Rohdatensatz enthält echte
Formatierungsfehler (`"$39.99"`, `"50% off"`, `"Jul 12th, 2020 8:09 pm"`), und der
Referenzdatensatz enthält für dieselben Zeilen bereits die korrekt geparste Form.
Das entspricht der Vorgehensweise, die auch in der Vorlagenarbeit (Merdan Kücük,
Experiment 2) für die Inkonsistenzbehebung verwendet wurde.

Zusätzlich werden `price` und `saving` per Regel (Regex) in Formatklassen eingeteilt
(vgl. `build_eda_notebook.py`) - diese Klassen dienen als Trainingslabel für den
XGBoost-Formatklassifikator im Experiment-Notebook.


In [7]:
def classify_price_format(s):
    s = str(s).strip()
    if re.fullmatch(r"\d+(\.\d+)?", s):
        return "numerisch"
    if re.fullmatch(r"\$\d+(\.\d+)?", s):
        return "dollar_prefix"
    if "%" in s:
        return "prozent"
    if "/" in s:
        return "mengenangabe"
    if re.search(r"(USD|CAD)", s, re.IGNORECASE):
        return "waehrungssuffix"
    if s.lower() in ("free", "varies"):
        return "wortwert"
    if "-" in s:
        return "preisspanne"
    return "sonstiges"

def classify_saving_format(s):
    s = str(s).strip()
    if "%" in s:
        return "prozent"
    if s.startswith("$"):
        return "dollarbetrag"
    if "off" in s.lower():
        return "wortkombination"
    if re.fullmatch(r"\d+(\.\d+)?", s):
        return "reine_zahl"
    return "sonstiges"

def build_format_benchmark(raw_col, clean_col, classify_fn, out_name, test_frac=0.30):
    left = df_raw[["row_id", raw_col]].rename(columns={"row_id": "row_id_raw", raw_col: "raw_value"})
    right = df_clean[["row_id", clean_col]].rename(columns={"row_id": "row_id_clean", clean_col: "true_clean_value"})
    merged = row_mapping.merge(left, on="row_id_raw").merge(right, on="row_id_clean")
    merged = merged[merged["raw_value"].notna() & merged["true_clean_value"].notna()].copy()
    merged["format_class"] = merged["raw_value"].apply(classify_fn)

    train, test = train_test_split(merged, test_size=test_frac, stratify=merged["format_class"],
                                    random_state=SEED)
    merged["split"] = "train"
    merged.loc[test.index, "split"] = "test"

    merged.to_csv(f"benchmark/{out_name}.csv", index=False)
    print(f"{out_name}: {len(merged)} Zeilen, Formatklassen: {merged['format_class'].value_counts().to_dict()}")
    return merged

_ = build_format_benchmark("price", "price", classify_price_format, "tf3_format_price")
_ = build_format_benchmark("saving", "saving", classify_saving_format, "tf3_format_saving")


tf3_format_price: 855 Zeilen, Formatklassen: {'numerisch': 511, 'dollar_prefix': 293, 'sonstiges': 33, 'waehrungssuffix': 7, 'mengenangabe': 4, 'prozent': 4, 'preisspanne': 3}
tf3_format_saving: 495 Zeilen, Formatklassen: {'prozent': 325, 'dollarbetrag': 127, 'reine_zahl': 22, 'sonstiges': 11, 'wortkombination': 10}


In [8]:
# --- Datumsspalten: eigener Parser fürs Rohformat, Referenzdatensatz als Ground Truth ---
date_rows = []
for col in ["creation_date", "expiry", "last_reply"]:
    left = df_raw[["row_id", col]].rename(columns={"row_id": "row_id_raw", col: "raw_value"})
    right = df_clean[["row_id", col]].rename(columns={"row_id": "row_id_clean", col: "true_clean_value"})
    merged = row_mapping.merge(left, on="row_id_raw").merge(right, on="row_id_clean")
    merged = merged[merged["raw_value"].notna() & merged["true_clean_value"].notna()].copy()
    merged["source_column"] = col
    date_rows.append(merged[["row_id_raw", "row_id_clean", "source_column", "raw_value", "true_clean_value"]])

date_bench = pd.concat(date_rows, ignore_index=True)

# Sanity-Check (Validierung der Ground-Truth-Daten, Abschnitt 3.2.4): stimmt unser eigener
# Rohformat-Parser mit dem Referenzwert überein? -> bestätigt, dass rfd_main_cleaned.csv
# als Ground Truth für die Datumsspalten vertrauenswürdig ist.
def parse_generic_raw_date(s):
    s2 = re.sub(r"(\d+)(st|nd|rd|th)", r"\1", str(s))
    for fmt in ("%b %d, %Y %I:%M %p", "%b %d, %Y"):
        try:
            return pd.to_datetime(s2, format=fmt)
        except ValueError:
            continue
    return pd.NaT

date_bench["self_parsed"] = date_bench["raw_value"].apply(parse_generic_raw_date)
date_bench["true_clean_parsed"] = pd.to_datetime(date_bench["true_clean_value"], errors="coerce")
agreement = (date_bench["self_parsed"].dt.floor("min") == date_bench["true_clean_parsed"].dt.floor("min")).mean()
print(f"Ground-Truth-Validierung Datumsspalten: {agreement*100:.1f}% Übereinstimmung eigener Parser vs. Referenzdatensatz")

train, test = train_test_split(date_bench, test_size=0.30, stratify=date_bench["source_column"], random_state=SEED)
date_bench["split"] = "train"
date_bench.loc[test.index, "split"] = "test"
date_bench = date_bench.drop(columns=["self_parsed", "true_clean_parsed"])
date_bench.to_csv("benchmark/tf3_format_date.csv", index=False)
print(f"tf3_format_date: {len(date_bench)} Zeilen gespeichert.")


Ground-Truth-Validierung Datumsspalten: 87.8% Übereinstimmung eigener Parser vs. Referenzdatensatz
tf3_format_date: 2950 Zeilen gespeichert.


## 4. TF4 - Semantische Heterogenität: Mehrheitsvotum-Mapping als Ground Truth

Für die 12 `parent_category`-Klassen und 55 `thread_category`-Werte existiert keine
offiziell "korrekte" Zuordnung. Als Ground Truth wird daher je `thread_category` die
**häufigste** zugehörige `parent_category` (Mehrheitsvotum) unter allen Zeilen mit
bekannten Werten bestimmt. Mehrdeutige `thread_category`-Werte (mehr als eine
zugehörige `parent_category`) werden separat ausgegeben und sollten vor der finalen
Verwendung manuell geprüft werden (Abschnitt 3.2.4, Validierung der Ground-Truth-Daten).


In [9]:
both = df_raw[df_raw["thread_category"].notna() & df_raw["parent_category"].notna()].copy()

mapping_counts = both.groupby("thread_category")["parent_category"].nunique().sort_values(ascending=False)
ambiguous = mapping_counts[mapping_counts > 1]

majority_map = (both.groupby("thread_category")["parent_category"]
                 .agg(lambda s: s.value_counts().idxmax())
                 .rename("majority_parent_category"))
vote_share = (both.groupby("thread_category")["parent_category"]
              .agg(lambda s: s.value_counts(normalize=True).max())
              .rename("vote_share"))

mapping_table = pd.concat([majority_map, vote_share], axis=1).reset_index()
mapping_table["is_ambiguous"] = mapping_table["thread_category"].isin(ambiguous.index)
mapping_table.to_csv("benchmark/tf4_thread_to_parent_mapping.csv", index=False)

print(f"thread_category-Werte gesamt: {mapping_table.shape[0]}, davon mehrdeutig: {mapping_table['is_ambiguous'].sum()}")
print("\n>>> BITTE PRÜFEN: benchmark/tf4_thread_to_parent_mapping.csv, Zeilen mit is_ambiguous=True <<<")
mapping_table[mapping_table["is_ambiguous"]]


thread_category-Werte gesamt: 42, davon mehrdeutig: 1

>>> BITTE PRÜFEN: benchmark/tf4_thread_to_parent_mapping.csv, Zeilen mit is_ambiguous=True <<<


,thread_category,majority_parent_category,vote_share,is_ambiguous
29,Other,Home & Garden,0.258065,True


In [10]:
# --- Evaluationsdatensatz: Zeilen mit bekannter parent_category, Train/Val/Test-Split ---
eval_set = both[["row_id", "thread_category", "title", "parent_category"]].rename(
    columns={"parent_category": "true_parent_category"})

train_val, test = train_test_split(eval_set, test_size=0.15, stratify=eval_set["true_parent_category"],
                                    random_state=SEED)
train, val = train_test_split(train_val, test_size=0.15/0.85, stratify=train_val["true_parent_category"],
                               random_state=SEED)
eval_set["split"] = "train"
eval_set.loc[val.index, "split"] = "val"
eval_set.loc[test.index, "split"] = "test"
eval_set.to_csv("benchmark/tf4_semantic_eval.csv", index=False)
print(f"tf4_semantic_eval: {len(eval_set)} Zeilen, Split: {eval_set['split'].value_counts().to_dict()}")

# --- Anwendungsmenge: parent_category fehlt, thread_category bekannt (für die finale Pipeline) ---
application_set = df_raw[df_raw["parent_category"].isna() & df_raw["thread_category"].notna()][
    ["row_id", "thread_category", "title"]]
application_set.to_csv("benchmark/tf4_application_set.csv", index=False)
print(f"tf4_application_set (für Kapitel 4.6, echte Lücken): {len(application_set)} Zeilen")


tf4_semantic_eval: 825 Zeilen, Split: {'train': 577, 'test': 124, 'val': 124}
tf4_application_set (für Kapitel 4.6, echte Lücken): 500 Zeilen


## 5. Zusammenfassung

Alle Benchmark- und Ground-Truth-Dateien liegen jetzt in `benchmark/`. Wichtig für
die nachfolgenden Experiment-Notebooks:

- **Nie** die `_groundtruth.csv`-Dateien oder die `true_*`-Spalten als Modell-Input verwenden.
- Alle Notebooks verwenden denselben `split`: `train` für klassische ML-Methoden zum
  Fitten, `val` optional zum Tunen, `test` für den finalen, berichteten Vergleich.
  Claude erhält **nur** die `test`-Zeilen (kein Training möglich/nötig, siehe Abschnitt 3.5.5).
- Jedes LLM-Notebook soll seine Laufzeit/Tokens/Kosten in `results/laufzeit_kosten_log.csv`
  anhängen (Spalten: experiment, method, n_items, wall_time_sec, input_tokens,
  output_tokens, estimated_cost_usd, model_name) - Grundlage für
  `Laufzeit_Kosten_Vergleich.ipynb`.


In [11]:
import glob
print("Erzeugte Benchmark-Dateien:")
for f in sorted(glob.glob("benchmark/*.csv")):
    n = sum(1 for _ in open(f)) - 1
    print(f"  {f:50s} {n:5d} Zeilen")


Erzeugte Benchmark-Dateien:
  benchmark/raw_clean_row_mapping.csv                 1293 Zeilen
  benchmark/tf1_duplicate_pairs.csv                    736 Zeilen
  benchmark/tf1_duplicate_pool.csv                    1476 Zeilen
  benchmark/tf2_missing_parent_category.csv            825 Zeilen
  benchmark/tf2_missing_parent_category_groundtruth.csv   248 Zeilen
  benchmark/tf2_missing_price.csv                     1084 Zeilen
  benchmark/tf2_missing_price_groundtruth.csv          326 Zeilen
  benchmark/tf3_format_date.csv                       2950 Zeilen
  benchmark/tf3_format_price.csv                       855 Zeilen
  benchmark/tf3_format_saving.csv                      495 Zeilen
  benchmark/tf4_application_set.csv                    500 Zeilen
  benchmark/tf4_semantic_eval.csv                      825 Zeilen
  benchmark/tf4_thread_to_parent_mapping.csv            42 Zeilen
